In [1]:


import torch
from transformers import AutoImageProcessor, AutoModel
from torchvision import transforms
import torch.nn.functional as F
import open_clip

from natsort import natsorted
from PIL import Image

from typing import List
import shutil
from pathlib import Path
from typing import List,Dict,Any,Tuple
import os


In [4]:
from dreamsim import dreamsim
import lpips

In [5]:

class CLIPPipeline:
    def __init__(
        self,
        model_name: str = "ViT-B-32",
        device: str = "cuda:0",
    ):
        self.device = torch.device(device)
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name, device=self.device
        )
        self.tokenizer = open_clip.get_tokenizer(model_name)
        
        self.model.eval()

    # --------- Utilities ---------
    def get_image_files(self, directory: str) -> List[str]:
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
        paths = [str(p) for p in Path(directory).rglob("*") if p.is_file() and p.suffix.lower() in exts]
        return natsorted(paths)

    # --------- Embedding ---------
    def get_clip_image_embedding(self, image_path: str) -> torch.Tensor:
        img = Image.open(image_path).convert("RGB")
        with torch.no_grad():
            x = self.preprocess(img).unsqueeze(0).to(self.device)  # [1, C, H, W]
            feat = self.model.encode_image(x)[0]                   # [D]
            feat = F.normalize(feat, p=2, dim=0)                   # [D], L2-normalized
        return feat


    def _get_clip_text_embeddings(self, texts: List[str]) -> torch.Tensor:
        """
        Returns:
            text_features: [N_text, D] L2-normalized
        """
        if not isinstance(texts, (list, tuple)) or len(texts) == 0:
            raise ValueError("texts must be a non-empty list of strings")

        with torch.no_grad():
            tokens = self.tokenizer(texts).to(self.device)      # [N_text, ctx]
            tfeat = self.model.encode_text(tokens)              # [N_text, D]
            tfeat = F.normalize(tfeat, p=2, dim=-1)             # [N_text, D]
        return tfeat
    
    # --------- Pairwise (single pair) ---------
    def compute_image_similarity(self, image_path1: str, image_path2: str) -> Dict[str, Any]:
        e1 = self.get_clip_image_embedding(image_path1)  # [D]
        e2 = self.get_clip_image_embedding(image_path2)  # [D]
        sim = F.cosine_similarity(e1, e2, dim=0).item()
        return {"similarity": sim, "embedding1": e1, "embedding2": e2}


    def compute_clip_image_pairwise_similarity(
        self,
        generated_path: str,
        ref_path: str,
        exclude_self: bool = True,
    ) -> Tuple[float, List[float]]:
        """
        For each generated image, compute cosine similarity to every reference image (pairwise),
        then average over references.

        Args:
            generated_path (str): Directory containing generated images.
            ref_path (str): Directory containing reference images.
            exclude_self (bool):
                If True, exclude pairs where the generated and reference image
                correspond to the same sample, determined by matching filenames.

        Returns:
            Tuple[float, List[float]]:
                average_similarity (float): Mean over all per-image mean similarities.
                similarities (List[float]): One mean similarity per generated image.
        """
        gen_paths = self.get_image_files(generated_path)
        ref_paths = self.get_image_files(ref_path)

        if not gen_paths:
            raise ValueError(f"No images found in generated directory: {generated_path}")
        if not ref_paths:
            raise ValueError(f"No images found in reference directory: {ref_path}")

        # Precompute reference embeddings and filenames
        ref_embs = []
        ref_names = []
        for rp in ref_paths:
            ref_embs.append(self.get_clip_image_embedding(rp))
            ref_names.append(Path(rp).name)

        ref_embs = torch.stack(ref_embs, dim=0)  # [N_ref, D]

        similarities: List[float] = []

        for gp in gen_paths:
            g = self.get_clip_image_embedding(gp)  # [D]
            g_name = Path(gp).name

            sims = F.cosine_similarity(g.unsqueeze(0), ref_embs, dim=-1)  # [N_ref]

            if exclude_self:
                mask = torch.tensor(
                    [g_name != r_name for r_name in ref_names],
                    dtype=torch.bool,
                    device=sims.device,
                )

                if mask.sum().item() == 0:
                    continue

                sims = sims[mask]

            similarities.append(sims.mean().item())

        if len(similarities) == 0:
            raise ValueError("No valid image pairs found after applying exclude_self.")

        average_similarity = sum(similarities) / len(similarities)
        return float(average_similarity), similarities


    # --------- Pairwise (folders) ---------
    # def compute_clip_image_pairwise_similarity(
    #     self,
    #     generated_path: str,
    #     ref_path: str,
    # ) -> Tuple[float, List[float]]:
    #     """
    #     For each generated image, compute cosine similarity to every reference image (pairwise),
    #     then average over references. Returns:
    #         average_similarity (float): mean over all per-image similarities
    #         similarities (List[float]): one mean similarity per generated image
    #     """
    #     gen_paths = self.get_image_files(generated_path)
    #     ref_paths = self.get_image_files(ref_path)
    #     if not gen_paths:
    #         raise ValueError(f"No images found in generated directory: {generated_path}")
    #     if not ref_paths:
    #         raise ValueError(f"No images found in reference directory: {ref_path}")

    #     # Precompu
    #     # te reference embeddings
    #     ref_embs = [self.get_clip_image_embedding(p) for p in ref_paths]
    #     ref_embs = torch.stack(ref_embs, dim=0)  # [N_ref, D]

    #     similarities: List[float] = []
    #     for gp in gen_paths:
    #         g = self.get_clip_image_embedding(gp)                 # [D]
    #         sims = F.cosine_similarity(g.unsqueeze(0), ref_embs, dim=-1)  # [N_ref]
    #         similarities.append(sims.mean().item())

    #     average_similarity = sum(similarities) / len(similarities)
    #     return float(average_similarity), similarities
    
    
    def _get_logit_scale(self) -> torch.Tensor:
        """
        open_clip stores logit scale as a learnable parameter (log space).
        We use exp() to get the multiplicative scale.
        """
        # model.logit_scale shape is typically [1]
        return self.model.logit_scale.exp()
    
        
    def compute_image_text_pairwise_similarity(
        self,
        generated_path: str,
        texts: List[str],
        batch_size: int = 64,
        return_probs: bool = True,
    ) -> Dict[str, Any]:
        """
        For each generated image, compute similarity to each text.
        Uses CLIP-style logits: logit_scale * image_features @ text_features.T

        Returns dict:
            {
            "image_paths": List[str],
            "texts": List[str],
            "logits": Tensor [N_img, N_text],
            "probs":  Tensor [N_img, N_text]  (if return_probs)
            }
        """
        gen_paths = self.get_image_files(generated_path)
        if not gen_paths:
            raise ValueError(f"No images found in generated directory: {generated_path}")

        if not isinstance(texts, (list, tuple)) or len(texts) == 0:
            raise ValueError("texts must be a non-empty list of strings")

        # Precompute text embeddings once (usually float32)
        text_features = self._get_clip_text_embeddings(texts)  # [N_text, D]
        logit_scale = self._get_logit_scale()                  # scalar-ish

        all_logits: List[torch.Tensor] = []
        use_amp = (self.device.type == "cuda")

        with torch.no_grad():
            for i in range(0, len(gen_paths), batch_size):
                batch_paths = gen_paths[i : i + batch_size]

                # Load + preprocess images
                imgs = []
                for p in batch_paths:
                    img = Image.open(p).convert("RGB")
                    imgs.append(self.preprocess(img))
                x = torch.stack(imgs, dim=0).to(self.device)  # [B, C, H, W]

                # Encode images (AMP optional)
                if use_amp:
                    with torch.autocast(device_type="cuda"):
                        img_feat = self.model.encode_image(x)  # [B, D] (likely fp16/bf16)
                else:
                    img_feat = self.model.encode_image(x)      # [B, D] (likely fp32)

                img_feat = F.normalize(img_feat, p=2, dim=-1)  # [B, D]

                # ---- IMPORTANT: align dtypes for matmul (fix Half vs Float) ----
                tf = text_features.to(dtype=img_feat.dtype)    # [N_text, D]
                ls = logit_scale.to(dtype=img_feat.dtype)      # scalar-ish

                logits = ls * (img_feat @ tf.T)                # [B, N_text]
                all_logits.append(logits)

        logits = torch.cat(all_logits, dim=0)  # [N_img, N_text]

        out: Dict[str, Any] = {
            "image_paths": gen_paths,
            "texts": list(texts),
            "logits": logits,
        }
        if return_probs:
            out["probs"] = logits.softmax(dim=-1)
        return out

    def compute_image_text_accuracy(
        self,
        generated_path: str,
        texts: List[str],
        gt_label: str,
        batch_size: int = 64,
    ) -> Dict[str, Any]:
        """
        - assert gt_label in texts
        - uses compute_image_text_pairwise_similarity
        - top1 prediction = argmax over texts
        - computes top1 accuracy across all images in generated_path
          assuming every image should match gt_label
        """
        assert gt_label in texts, f"gt_label='{gt_label}' must be in texts"

        res = self.compute_image_text_pairwise_similarity(
            generated_path=generated_path,
            texts=texts,
            batch_size=batch_size,
            return_probs=False,
        )

        logits: torch.Tensor = res["logits"]  # [N_img, N_text]
        pred_idx = logits.argmax(dim=-1)      # [N_img]
        pred_labels = [texts[i] for i in pred_idx.tolist()]

        correct = [(p == gt_label) for p in pred_labels]
        acc = sum(correct) / len(correct) if len(correct) > 0 else 0.0

        return {
            "accuracy_top1": float(acc),
            "gt_label": gt_label,
            "pred_labels": pred_labels,
            "image_paths": res["image_paths"],
            "texts": res["texts"],
            "logits": logits,
        }
        
        
    
# class DINOPipeline:
#     def __init__(
#         self,
#         model_name: str = "facebook/dinov2-base",
#         device: str = "cuda:0",
#     ):
#         self.device = torch.device(device)
#         self.model = AutoModel.from_pretrained(model_name)
#         self.model.to(self.device)
#         self.model.eval()
        
#         # DINO transforms (standard ImageNet preprocessing)
#         self.preprocess = transforms.Compose([
#             transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
#             transforms.CenterCrop(224),
#             transforms.ToTensor(),
#             transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
#         ])

#     # --------- Utilities ---------
#     def get_image_files(self, directory: str) -> List[str]:
#         exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
#         paths = [str(p) for p in Path(directory).rglob("*") if p.is_file() and p.suffix.lower() in exts]
#         return natsorted(paths)

#     # --------- Embedding ---------
#     def get_dino_image_embedding(self, image_path: str) -> torch.Tensor:
#         img = Image.open(image_path).convert("RGB")
#         with torch.no_grad():
#             x = self.preprocess(img).unsqueeze(0).to(self.device)  # [1, C, H, W]
#             outputs = self.model(x)
#             # Extract CLS token embedding from last hidden state
#             feat = outputs.last_hidden_state[0, 0]  # [D], CLS token
#             feat = F.normalize(feat, p=2, dim=0)     # [D], L2-normalized
#         return feat

#     # --------- Pairwise (single pair) ---------
#     def compute_image_similarity(self, image_path1: str, image_path2: str) -> Dict[str, Any]:
#         e1 = self.get_dino_image_embedding(image_path1)  # [D]
#         e2 = self.get_dino_image_embedding(image_path2)  # [D]
#         sim = F.cosine_similarity(e1, e2, dim=0).item()
#         return {"similarity": sim, "embedding1": e1, "embedding2": e2}


#     # --------- Pairwise (folders) ---------
#     def compute_dino_image_pairwise_similarity(
#         self,
#         generated_path: str,
#         ref_path: str,
#         exclude_self: bool = True,
#     ) -> Tuple[float, List[float]]:
#         """
#         For each generated image, compute cosine similarity to every reference image (pairwise),
#         then average over references.

#         Args:
#             generated_path (str): Directory containing generated images.
#             ref_path (str): Directory containing reference images.
#             exclude_self (bool):
#                 If True, exclude pairs where the generated and reference image
#                 correspond to the same sample, determined by matching filenames.

#         Returns:
#             Tuple[float, List[float]]:
#                 average_similarity (float): Mean over all per-image mean similarities.
#                 similarities (List[float]): One mean similarity per generated image.
#         """
#         gen_paths = self.get_image_files(generated_path)
#         ref_paths = self.get_image_files(ref_path)

#         if not gen_paths:
#             raise ValueError(f"No images found in generated directory: {generated_path}")
#         if not ref_paths:
#             raise ValueError(f"No images found in reference directory: {ref_path}")

#         # Precompute reference embeddings + filenames
#         ref_embs = []
#         ref_names = []
#         for rp in ref_paths:
#             ref_embs.append(self.get_dino_image_embedding(rp))
#             ref_names.append(Path(rp).name)

#         ref_embs = torch.stack(ref_embs, dim=0)  # [N_ref, D]

#         similarities: List[float] = []

#         for gp in gen_paths:
#             g = self.get_dino_image_embedding(gp)  # [D]
#             g_name = Path(gp).name

#             sims = F.cosine_similarity(g.unsqueeze(0), ref_embs, dim=-1)  # [N_ref]

#             if exclude_self:
#                 mask = torch.tensor(
#                     [g_name != r_name for r_name in ref_names],
#                     dtype=torch.bool,
#                     device=sims.device,
#                 )

#                 if mask.sum().item() == 0:
#                     continue

#                 sims = sims[mask]

#             similarities.append(sims.mean().item())

#         if len(similarities) == 0:
#             raise ValueError("No valid image pairs found after applying exclude_self.")

#         average_similarity = sum(similarities) / len(similarities)
#         return float(average_similarity), similarities


#     # # --------- Pairwise (folders) ---------
#     # def compute_dino_image_pairwise_similarity(
#     #     self,
#     #     generated_path: str,
#     #     ref_path: str,
#     # ) -> Tuple[float, List[float]]:
#     #     """
#     #     For each generated image, compute cosine similarity to every reference image (pairwise),
#     #     then average over references. Returns:
#     #         average_similarity (float): mean over all per-image similarities
#     #         similarities (List[float]): one mean similarity per generated image
#     #     """
#     #     gen_paths = self.get_image_files(generated_path)
#     #     ref_paths = self.get_image_files(ref_path)
#     #     if not gen_paths:
#     #         raise ValueError(f"No images found in generated directory: {generated_path}")
#     #     if not ref_paths:
#     #         raise ValueError(f"No images found in reference directory: {ref_path}")

#     #     # Precompute reference embeddings
#     #     ref_embs = [self.get_dino_image_embedding(p) for p in ref_paths]
#     #     ref_embs = torch.stack(ref_embs, dim=0)  # [N_ref, D]

#     #     similarities: List[float] = []
#     #     for gp in gen_paths:
#     #         g = self.get_dino_image_embedding(gp)                 # [D]
#     #         sims = F.cosine_similarity(g.unsqueeze(0), ref_embs, dim=-1)  # [N_ref]
#     #         similarities.append(sims.mean().item())

#     #     average_similarity = sum(similarities) / len(similarities)
#     #     return float(average_similarity), similarities
    

In [6]:
# patch sim
class DINOPipeline:
    def __init__(
        self,
        model_name: str = "facebook/dinov2-base",
        device: str = "cuda:0",
    ):
        self.device = torch.device(device)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

        self.preprocess = transforms.Compose([
            transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ])

    # --------- Utilities ---------
    def get_image_files(self, directory: str) -> List[str]:
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
        paths = [str(p) for p in Path(directory).rglob("*") if p.is_file() and p.suffix.lower() in exts]
        return natsorted(paths)

    # --------- Global embedding (CLS token) ---------
    def get_dino_image_embedding(self, image_path: str) -> torch.Tensor:
        img = Image.open(image_path).convert("RGB")
        with torch.no_grad():
            x = self.preprocess(img).unsqueeze(0).to(self.device)   # [1, C, H, W]
            outputs = self.model(pixel_values=x)
            feat = outputs.last_hidden_state[0, 0]                  # [D] CLS token
            feat = F.normalize(feat, p=2, dim=0)
        return feat

    # --------- Patch embedding ---------
    def get_dino_patch_image_embedding(self, image_path: str) -> torch.Tensor:
        """
        Returns:
            patch_feat: [P, D]
                L2-normalized patch-token embeddings.
                Excludes the CLS token.
        """
        img = Image.open(image_path).convert("RGB")
        with torch.no_grad():
            x = self.preprocess(img).unsqueeze(0).to(self.device)   # [1, C, H, W]
            outputs = self.model(pixel_values=x)
            patch_feat = outputs.last_hidden_state[0, 1:, :]        # [P, D], exclude CLS
            patch_feat = F.normalize(patch_feat, p=2, dim=-1)       # normalize each patch
        return patch_feat

    # --------- Pairwise (single pair, global CLS) ---------
    def compute_image_similarity(self, image_path1: str, image_path2: str) -> Dict[str, Any]:
        e1 = self.get_dino_image_embedding(image_path1)
        e2 = self.get_dino_image_embedding(image_path2)
        sim = F.cosine_similarity(e1, e2, dim=0).item()
        return {"similarity": sim, "embedding1": e1, "embedding2": e2}

    # --------- Pairwise (folders, global CLS) ---------
    def compute_dino_image_pairwise_similarity(
        self,
        generated_path: str,
        ref_path: str,
        exclude_self: bool = True,
    ) -> Tuple[float, List[float]]:
        gen_paths = self.get_image_files(generated_path)
        ref_paths = self.get_image_files(ref_path)

        if not gen_paths:
            raise ValueError(f"No images found in generated directory: {generated_path}")
        if not ref_paths:
            raise ValueError(f"No images found in reference directory: {ref_path}")

        ref_embs = []
        ref_names = []
        for rp in ref_paths:
            ref_embs.append(self.get_dino_image_embedding(rp))
            ref_names.append(Path(rp).name)

        ref_embs = torch.stack(ref_embs, dim=0)  # [N_ref, D]
        similarities: List[float] = []

        for gp in gen_paths:
            g = self.get_dino_image_embedding(gp)  # [D]
            g_name = Path(gp).name

            sims = F.cosine_similarity(g.unsqueeze(0), ref_embs, dim=-1)  # [N_ref]

            if exclude_self:
                mask = torch.tensor(
                    [g_name != r_name for r_name in ref_names],
                    dtype=torch.bool,
                    device=sims.device,
                )
                if mask.sum().item() == 0:
                    continue
                sims = sims[mask]

            similarities.append(sims.mean().item())

        if len(similarities) == 0:
            raise ValueError("No valid image pairs found after applying exclude_self.")

        average_similarity = sum(similarities) / len(similarities)
        return float(average_similarity), similarities

    # --------- Pairwise (folders, patch-level) ---------
    def compute_dino_patch_image_pairwise_similarity(
        self,
        generated_path: str,
        ref_path: str,
        exclude_self: bool = True,
    ) -> Tuple[float, List[float]]:
        """
        For each generated image, compute patch-level cosine similarity to every reference image,
        then average over references.

        Patch-level similarity for one image pair is computed by:
            1. extracting patch tokens [P, D] from both images
            2. cosine similarity for corresponding patches
            3. averaging across patches

        Returns:
            average_similarity (float): mean over all per-image mean similarities
            similarities (List[float]): one mean similarity per generated image
        """
        gen_paths = self.get_image_files(generated_path)
        ref_paths = self.get_image_files(ref_path)

        if not gen_paths:
            raise ValueError(f"No images found in generated directory: {generated_path}")
        if not ref_paths:
            raise ValueError(f"No images found in reference directory: {ref_path}")

        # Precompute reference patch embeddings + filenames
        ref_patch_embs = []
        ref_names = []
        for rp in ref_paths:
            ref_patch_embs.append(self.get_dino_patch_image_embedding(rp))  # [P, D]
            ref_names.append(Path(rp).name)

        similarities: List[float] = []

        for gp in gen_paths:
            g_patch = self.get_dino_patch_image_embedding(gp)  # [P, D]
            g_name = Path(gp).name

            pair_sims = []
            for r_name, r_patch in zip(ref_names, ref_patch_embs):
                if exclude_self and g_name == r_name:
                    continue

                if g_patch.shape != r_patch.shape:
                    raise ValueError(
                        f"Patch shape mismatch between {gp} and reference image {r_name}: "
                        f"{tuple(g_patch.shape)} vs {tuple(r_patch.shape)}"
                    )

                # cosine for corresponding patches, then mean over patches
                sim_patches = F.cosine_similarity(g_patch, r_patch, dim=-1)  # [P]
                pair_sims.append(sim_patches.mean().item())

            if len(pair_sims) == 0:
                continue

            similarities.append(sum(pair_sims) / len(pair_sims))

        if len(similarities) == 0:
            raise ValueError("No valid image pairs found after applying exclude_self.")

        average_similarity = sum(similarities) / len(similarities)
        return float(average_similarity), similarities

In [7]:
   
    
class DreamSimPipeline:
    def __init__(
        self,
        device: str = "cuda:0",
    ):
        self.device = torch.device(device)
        self.model, self.preprocess = dreamsim(pretrained=True, dreamsim_type="dino_vitb16", device=device)

        self.model.to(self.device)
        self.model.eval()

    # --------- Utilities ---------
    def get_image_files(self, directory: str) -> List[str]:
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
        paths = [str(p) for p in Path(directory).rglob("*") if p.is_file() and p.suffix.lower() in exts]
        return natsorted(paths)

    # --------- Embedding ---------
    def get_image_embedding(self, image_path: str) -> torch.Tensor:
        
        img = self.preprocess(Image.open(image_path)).to(self.device)
        with torch.no_grad():
            embedding = self.model.embed(img)
        return embedding


    # --------- Pairwise (single pair) ---------
    def compute_image_similarity(self, image_path1: str, image_path2: str) -> Dict[str, Any]:
        e1 = self.get_image_embedding(image_path1)  # [D]
        e2 = self.get_image_embedding(image_path2)  # [D]
        sim = F.cosine_similarity(e1, e2, dim=0).item()
        return {"similarity": sim, "embedding1": e1, "embedding2": e2}

    # https://github.com/ssundaram21/dreamsim/blob/main/dreamsim/model.py
    # --------- Pairwise (folders) ---------
    def compute_image_pairwise_similarity(
        self,
        generated_path: str,
        ref_path: str,
        exclude_self: bool = True,
    ) -> Tuple[float, List[float]]:
        """
        For each generated image, compute cosine similarity to every reference image (pairwise),
        then average over references.

        Args:
            generated_path (str): Directory containing generated images.
            ref_path (str): Directory containing reference images.
            exclude_self (bool):
                If True, exclude pairs where the generated and reference image
                correspond to the same sample, determined by matching filenames.

        Returns:
            Tuple[float, List[float]]:
                average_similarity (float): Mean over all per-image mean similarities.
                similarities (List[float]): One mean similarity per generated image.
        """
        gen_paths = self.get_image_files(generated_path)
        ref_paths = self.get_image_files(ref_path)

        if not gen_paths:
            raise ValueError(f"No images found in generated directory: {generated_path}")
        if not ref_paths:
            raise ValueError(f"No images found in reference directory: {ref_path}")

        # Precompute reference embeddings + filenames
        ref_embs = []
        ref_names = []
        for rp in ref_paths:
            ref_embs.append(self.get_image_embedding(rp))
            ref_names.append(Path(rp).name)

        ref_embs = torch.stack(ref_embs, dim=0)  # [N_ref, D]

        similarities: List[float] = []

        for gp in gen_paths:
            g = self.get_image_embedding(gp)  # [D]
            g_name = Path(gp).name

            sims = F.cosine_similarity(g.unsqueeze(0), ref_embs, dim=-1)  # [N_ref]

            if exclude_self:
                mask = torch.tensor(
                    [g_name != r_name for r_name in ref_names],
                    dtype=torch.bool,
                    device=sims.device,
                )

                if mask.sum().item() == 0:
                    continue

                sims = sims[mask]

            similarities.append(sims.mean().item())

        if len(similarities) == 0:
            raise ValueError("No valid image pairs found after applying exclude_self.")

        average_similarity = sum(similarities) / len(similarities)
        return float(average_similarity), similarities

In [8]:

class LPIPSPipeline:
    def __init__(
        self,
        net: str = "vgg",
        device: str = "cuda:0",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.model = lpips.LPIPS(net=net)
        self.model.to(self.device)
        self.model.eval()

    # --------- Utilities ---------
    def get_image_files(self, directory: str) -> List[str]:
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
        paths = [
            str(p) for p in Path(directory).rglob("*")
            if p.is_file() and p.suffix.lower() in exts
        ]
        return natsorted(paths)

    def load_lpips_image(self, image_path: str) -> torch.Tensor:
        """
        Load image as RGB and convert to LPIPS tensor format: [1, 3, H, W], range [-1, 1].
        """
        img = Image.open(image_path).convert("RGB")
        img_tensor = lpips.im2tensor(lpips.load_image(image_path))  # [1, 3, H, W], [-1, 1]
        return img_tensor.to(self.device)

    # --------- Single pair ---------
    def compute_image_similarity(self, image_path1: str, image_path2: str) -> Dict[str, Any]:
        """
        Compute LPIPS distance for one image pair.
        Lower is more similar.
        """
        img1 = self.load_lpips_image(image_path1)
        img2 = self.load_lpips_image(image_path2)

        with torch.no_grad():
            dist = self.model(img1, img2).item()

        return {
            "lpips_distance": float(dist),
            "image_path1": image_path1,
            "image_path2": image_path2,
        }

    # --------- Pairwise (folders) ---------
    def compute_image_pairwise_lpips(
        self,
        generated_path: str,
        ref_path: str,
        exclude_self: bool = True,
    ) -> Tuple[float, List[float]]:
        """
        For each generated image, compute LPIPS distance to every reference image,
        then average over references.

        Args:
            generated_path (str): Directory containing generated images.
            ref_path (str): Directory containing reference images.
            exclude_self (bool):
                If True, exclude pairs where the generated and reference image
                correspond to the same sample, determined by matching filenames.

        Returns:
            Tuple[float, List[float]]:
                average_distance (float): Mean over all per-image mean distances.
                distances (List[float]): One mean LPIPS distance per generated image.

        Note:
            Lower LPIPS means more perceptually similar.
        """
        gen_paths = self.get_image_files(generated_path)
        ref_paths = self.get_image_files(ref_path)

        if not gen_paths:
            raise ValueError(f"No images found in generated directory: {generated_path}")
        if not ref_paths:
            raise ValueError(f"No images found in reference directory: {ref_path}")

        # Preload reference images + filenames
        ref_imgs = []
        ref_names = []
        for rp in ref_paths:
            ref_imgs.append(self.load_lpips_image(rp))
            ref_names.append(Path(rp).name)

        distances: List[float] = []

        with torch.no_grad():
            for gp in gen_paths:
                g = self.load_lpips_image(gp)
                g_name = Path(gp).name

                pair_dists = []

                for r, r_name in zip(ref_imgs, ref_names):
                    if exclude_self and g_name == r_name:
                        continue

                    d = self.model(g, r).item()
                    pair_dists.append(d)

                if len(pair_dists) == 0:
                    continue

                distances.append(sum(pair_dists) / len(pair_dists))

        if len(distances) == 0:
            raise ValueError("No valid image pairs found after applying exclude_self.")

        average_distance = sum(distances) / len(distances)
        return float(average_distance), distances

In [9]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dino_pipeline = DINOPipeline(device=device)


clip_pipeline = CLIPPipeline(device=device)


dreamsim_pipeline = DreamSimPipeline(device=device)
lpips_pipeline = LPIPSPipeline(device=device)

Using cache found in ./models/facebookresearch_dino_main


Using cached ./models


/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:134: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in 

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth


/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_pat

In [7]:
image_path = 'data_root/generated/study/original_pretrained_sd1.4_bf16/a photo of dalmatian dog/7.50/'

avg_similaritiy ,similarities= dino_pipeline.compute_dino_image_pairwise_similarity(image_path, image_path)
print(avg_similaritiy)


0.783819603919983


In [8]:
# example DINO
base_dir = '/home/nessessence/mnt_tl_vision5/data/bright/DiffSynth-Studio/static/images'


our_path = os.path.join(base_dir, 'our_dog_zimg')
our_avg_similarity ,our_similarities= dino_pipeline.compute_dino_image_pairwise_similarity(our_path, our_path)


our_avg_cos_dist = 1 - our_avg_similarity

base_path = os.path.join(base_dir, 'base_dog_zimg')
base_avg_similarity ,base_similarities= dino_pipeline.compute_dino_image_pairwise_similarity(base_path, base_path)

base_avg_cos_dist = 1 - base_avg_similarity

# print(f"our_avg_similarity: {our_avg_similarity}")
# print(f"base_avg_similarity: {base_avg_similarity}")
print(f"our_avg_cos_dist: {our_avg_cos_dist}")
print(f"base_avg_cos_dist: {base_avg_cos_dist}")


our_avg_cos_dist: 0.7688966148843368
base_avg_cos_dist: 0.6681192204356193


In [11]:
model = 'zimg'

base_dir = '/home/nessessence/mnt_tl_vision5/data/bright/DiffSynth-Studio/static/images'

filtered_folders = [f for f in os.listdir(base_dir) if model in f]

# filtered_concepts = set([p.split('_')[1] for p in filtered_folders])

# not include split[0] and split[-1]
filtered_concepts = natsorted(list(set([ '_'.join(p.split('_')[1:-1]) for p in filtered_folders ])))

print

print(filtered_folders)
print(filtered_concepts)


print(f'model {model}')
for c in filtered_concepts:
    # print(f"Concept: {c}")
    our_path = os.path.join(base_dir, f'our_{c}_{model}')
    base_path = os.path.join(base_dir, f'base_{c}_{model}')
    
    if not os.path.exists(our_path) or not os.path.exists(base_path):
        print(f"  Skipping concept '{c}' because one of the folders does not exist.")
        continue
    
    # DINO similarity
    # our_dino_avg_similarity ,our_dino_similarities= dino_pipeline.compute_dino_image_pairwise_similarity(our_path, our_path)
    # our_dino_avg_cos_dist = 1 - our_dino_avg_similarity

    # base_dino_avg_similarity ,base_dino_similarities= dino_pipeline.compute_dino_image_pairwise_similarity(base_path, base_path)
    # base_dino_avg_cos_dist = 1 - base_dino_avg_similarity


    our_dino_avg_similarity ,our_dino_similarities= dino_pipeline.compute_dino_patch_image_pairwise_similarity(our_path, our_path)
    our_dino_avg_cos_dist = 1 - our_dino_avg_similarity

    base_dino_avg_similarity ,base_dino_similarities= dino_pipeline.compute_dino_patch_image_pairwise_similarity(base_path, base_path)
    base_dino_avg_cos_dist = 1 - base_dino_avg_similarity

    
    
    # CLIP similarity
    our_clip_avg_similarity ,our_clip_similarities= clip_pipeline.compute_clip_image_pairwise_similarity(our_path, our_path)
    our_clip_avg_cos_dist = 1 - our_clip_avg_similarity

    base_clip_avg_similarity ,base_clip_similarities= clip_pipeline.compute_clip_image_pairwise_similarity(base_path, base_path)
    base_clip_avg_cos_dist = 1 - base_clip_avg_similarity

    
    
    
    our_dreamsim_avg_similarity ,our_dreamsim_similarities= dreamsim_pipeline.compute_image_pairwise_similarity(our_path, our_path)
    our_dreamsim_avg_cos_dist = 1 - our_dreamsim_avg_similarity

    base_dreamsim_avg_similarity ,base_dreamsim_similarities= dreamsim_pipeline.compute_image_pairwise_similarity(base_path, base_path)
    base_dreamsim_avg_cos_dist = 1 - base_dreamsim_avg_similarity
    
    
    
    our_lpips_avg_distance, our_lpips_distances = lpips_pipeline.compute_image_pairwise_lpips(our_path, our_path)
    base_lpips_avg_distance, base_lpips_distances = lpips_pipeline.compute_image_pairwise_lpips(base_path, base_path)
    
    print(f'Concept: {c}')
    
    
    print(f"DINO: Our:{our_dino_avg_cos_dist:.4f} vs Base:{base_dino_avg_cos_dist:.4f}")
    
    print(f"CLIP: Our:{our_clip_avg_cos_dist:.4f} vs Base:{base_clip_avg_cos_dist:.4f}")
    
    print(f"DreamSim: Our:{our_dreamsim_avg_cos_dist:.4f} vs Base:{base_dreamsim_avg_cos_dist:.4f}")
    
    print(f"LPIPS: Our:{our_lpips_avg_distance:.4f} vs Base:{base_lpips_avg_distance:.4f}")
    


['our_dog_zimg', 'our_chinese_citizen_zimg', 'our_thai_temple_zimg', 'base_us_citizen_zimg', 'base_chinese_citizen_zimg', 'base_dof_zimg', 'base_doctor_zimg', 'base_thai_temple_zimg', 'our_us_citizen_zimg', 'base_dog_zimg', 'our_doctor_zimg']
['chinese_citizen', 'doctor', 'dof', 'dog', 'thai_temple', 'us_citizen']
model zimg
Concept: chinese_citizen
DINO: Our:0.7221 vs Base:0.5777
CLIP: Our:0.1724 vs Base:0.2192
DreamSim: Our:0.4896 vs Base:0.3269
LPIPS: Our:0.7700 vs Base:0.6592
Concept: doctor
DINO: Our:0.7871 vs Base:0.7277
CLIP: Our:0.1766 vs Base:0.0335
DreamSim: Our:0.5413 vs Base:0.4580
LPIPS: Our:0.7871 vs Base:0.6882
  Skipping concept 'dof' because one of the folders does not exist.
Concept: dog
DINO: Our:0.7859 vs Base:0.7500
CLIP: Our:0.1287 vs Base:0.1368
DreamSim: Our:0.5851 vs Base:0.4695
LPIPS: Our:0.7922 vs Base:0.7390
Concept: thai_temple
DINO: Our:0.7058 vs Base:0.5618
CLIP: Our:0.1665 vs Base:0.0904
DreamSim: Our:0.5113 vs Base:0.3209
LPIPS: Our:0.7873 vs Base:0.600

In [15]:
pwd

'/home/nessessence/mnt_tl_vision16/home/nessessence/uul'

In [16]:
model = 'sd1.4'

base_dir = 'data_root/generated/study/original_pretrained_sd1.5_bf16/'

filtered_folders = [f for f in os.listdir(base_dir)]

# filtered_concepts = set([p.split('_')[1] for p in filtered_folders])

# not include split[0] and split[-1]
# filtered_concepts = natsorted(list(set([ '_'.join(p.split('_')[1:-1]) for p in filtered_folders ])))
filtered_concepts = filtered_folders



print(f'model {model}')
for c in filtered_concepts:
    # print(f"Concept: {c}")
    # our_path = os.path.join(base_dir, f'our_{c}_{model}')
    base_path = os.path.join(base_dir, f'{c}','7.50')
    
    # if not os.path.exists(our_path) or not os.path.exists(base_path):
    #     print(f"  Skipping concept '{c}' because one of the folders does not exist.")
    #     continue
    
    # DINO similarity
    # our_dino_avg_similarity ,our_dino_similarities= dino_pipeline.compute_dino_image_pairwise_similarity(our_path, our_path)
    # our_dino_avg_cos_dist = 1 - our_dino_avg_similarity

    # base_dino_avg_similarity ,base_dino_similarities= dino_pipeline.compute_dino_image_pairwise_similarity(base_path, base_path)
    # base_dino_avg_cos_dist = 1 - base_dino_avg_similarity



    base_dino_avg_similarity ,base_dino_similarities= dino_pipeline.compute_dino_patch_image_pairwise_similarity(base_path, base_path)
    base_dino_avg_cos_dist = 1 - base_dino_avg_similarity

    
    
    # CLIP similarity


    base_clip_avg_similarity ,base_clip_similarities= clip_pipeline.compute_clip_image_pairwise_similarity(base_path, base_path)
    base_clip_avg_cos_dist = 1 - base_clip_avg_similarity

    


    base_dreamsim_avg_similarity ,base_dreamsim_similarities= dreamsim_pipeline.compute_image_pairwise_similarity(base_path, base_path)
    base_dreamsim_avg_cos_dist = 1 - base_dreamsim_avg_similarity
    
    
    
    base_lpips_avg_distance, base_lpips_distances = lpips_pipeline.compute_image_pairwise_lpips(base_path, base_path)
    
    print(f'Concept: {c}')
    
    
    print(f"DINO: Base:{base_dino_avg_cos_dist:.4f}")
    
    print(f"CLIP: Base:{base_clip_avg_cos_dist:.4f}")
    
    print(f"DreamSim: Base:{base_dreamsim_avg_cos_dist:.4f}")
    
    print(f"LPIPS: Base:{base_lpips_avg_distance:.4f}")
    


model sd1.4
Concept: an eagle
DINO: Base:0.7657
CLIP: Base:0.2742
DreamSim: Base:0.3792
LPIPS: Base:0.7589
Concept: a dog
DINO: Base:0.7855
CLIP: Base:0.1749
DreamSim: Base:0.5312
LPIPS: Base:0.7724
Concept: a portrait of us citizen
DINO: Base:0.7960
CLIP: Base:0.1893
DreamSim: Base:0.6164
LPIPS: Base:0.8201
Concept: a cat
DINO: Base:0.6438
CLIP: Base:0.1463
DreamSim: Base:0.2892
LPIPS: Base:0.7442


In [14]:
filtered_concepts

[]

In [ ]:

# DINO is not patch sim

model = 'zimg'

base_dir = '/home/nessessence/mnt_tl_vision5/data/bright/DiffSynth-Studio/static/images'

filtered_folders = [f for f in os.listdir(base_dir) if model in f]

# filtered_concepts = set([p.split('_')[1] for p in filtered_folders])

# not include split[0] and split[-1]
filtered_concepts = set([ '_'.join(p.split('_')[1:-1]) for p in filtered_folders ])

print

print(filtered_folders)
print(filtered_concepts)


print(f'model {model}')
for c in filtered_concepts:
    # print(f"Concept: {c}")
    our_path = os.path.join(base_dir, f'our_{c}_{model}')
    base_path = os.path.join(base_dir, f'base_{c}_{model}')
    
    if not os.path.exists(our_path) or not os.path.exists(base_path):
        print(f"  Skipping concept '{c}' because one of the folders does not exist.")
        continue
    
    # DINO similarity
    our_dino_avg_similarity ,our_dino_similarities= dino_pipeline.compute_dino_image_pairwise_similarity(our_path, our_path)
    our_dino_avg_cos_dist = 1 - our_dino_avg_similarity

    base_dino_avg_similarity ,base_dino_similarities= dino_pipeline.compute_dino_image_pairwise_similarity(base_path, base_path)
    base_dino_avg_cos_dist = 1 - base_dino_avg_similarity

    
    # CLIP similarity
    our_clip_avg_similarity ,our_clip_similarities= clip_pipeline.compute_clip_image_pairwise_similarity(our_path, our_path)
    our_clip_avg_cos_dist = 1 - our_clip_avg_similarity

    base_clip_avg_similarity ,base_clip_similarities= clip_pipeline.compute_clip_image_pairwise_similarity(base_path, base_path)
    base_clip_avg_cos_dist = 1 - base_clip_avg_similarity

    
    
    
    our_dreamsim_avg_similarity ,our_dreamsim_similarities= dreamsim_pipeline.compute_image_pairwise_similarity(our_path, our_path)
    our_dreamsim_avg_cos_dist = 1 - our_dreamsim_avg_similarity

    base_dreamsim_avg_similarity ,base_dreamsim_similarities= dreamsim_pipeline.compute_image_pairwise_similarity(base_path, base_path)
    base_dreamsim_avg_cos_dist = 1 - base_dreamsim_avg_similarity
    
    
    
    our_lpips_avg_distance, our_lpips_distances = lpips_pipeline.compute_image_pairwise_lpips(our_path, our_path)
    base_lpips_avg_distance, base_lpips_distances = lpips_pipeline.compute_image_pairwise_lpips(base_path, base_path)
    
    print(f'Concept: {c}')
    
    
    print(f"DINO: Our:{our_dino_avg_cos_dist:.4f} vs Base:{base_dino_avg_cos_dist:.4f}")
    
    print(f"CLIP: Our:{our_clip_avg_cos_dist:.4f} vs Base:{base_clip_avg_cos_dist:.4f}")
    
    print(f"DreamSim: Our:{our_dreamsim_avg_cos_dist:.4f} vs Base:{base_dreamsim_avg_cos_dist:.4f}")
    
    print(f"LPIPS: Our:{our_lpips_avg_distance:.4f} vs Base:{base_lpips_avg_distance:.4f}")
    


['our_dog_zimg', 'our_chinese_citizen_zimg', 'our_thai_temple_zimg', 'base_us_citizen_zimg', 'base_chinese_citizen_zimg', 'base_dof_zimg', 'base_doctor_zimg', 'base_thai_temple_zimg', 'our_us_citizen_zimg', 'base_dog_zimg', 'our_doctor_zimg']
{'thai_temple', 'doctor', 'us_citizen', 'dog', 'dof', 'chinese_citizen'}
model zimg
Concept: thai_temple
DINO: Our:0.3488 vs Base:0.2560
CLIP: Our:0.1509 vs Base:0.0890
DreamSim: Our:0.5113 vs Base:0.3209
LPIPS: Our:0.7873 vs Base:0.6002
Concept: doctor
DINO: Our:0.4170 vs Base:0.2982
CLIP: Our:0.1652 vs Base:0.0349
DreamSim: Our:0.5413 vs Base:0.4580
LPIPS: Our:0.7871 vs Base:0.6882
Concept: us_citizen
DINO: Our:0.6545 vs Base:0.3976
CLIP: Our:0.1532 vs Base:0.2250
DreamSim: Our:0.4754 vs Base:0.2984
LPIPS: Our:0.7603 vs Base:0.6569
Concept: dog
DINO: Our:0.7689 vs Base:0.6681
CLIP: Our:0.1166 vs Base:0.1105
DreamSim: Our:0.5851 vs Base:0.4695
LPIPS: Our:0.7922 vs Base:0.7390
  Skipping concept 'dof' because one of the folders does not exist.
Con

In [ ]:
filtered_concepts



{'a_bacteria',
 'a_dog',
 'a_panda',
 'an_eagle',
 'cat_flux',
 'corona_virus',
 'eagle_cfg1_scale4_p20',
 'eagle_cfg1_scale4_p5',
 'neuschwanstein_castle'}

In [57]:
ls  '/home/nessessence/mnt_tl_vision5/data/bright/DiffSynth-Studio/static/images/'

base_a_bacteria_flux/             our_a_dog_flux/
base_a_dog_flux/                  our_an_eagle_flux/
base_an_eagle_flux/               our_a_panda_flux/
base_a_panda_flux/                our_Barack_Obama_vid/
base_Barack_Obama_vid/            our_boat_vid/
base_boat_vid/                    our_burito_vid_large/
base_burito_vid_large/            our_cat_fight_interpolate_vid/
base_cat_edgemap_vid/             our_cat_flux_inpaint/
base_cat_flux_inpaint/            our_cat_grey_vid/
base_cat_grey_vid/                our_cat_mod_vid/
base_cat_mod_vid/                 our_chinese_citizen_zimg/
base_chinese_citizen_zimg/        our_doctor_zimg/
base_corona_virus_flux/           our_dog_vid/
base_doctor_zimg/                 our_dog_zimg/
base_dof_zimg/                    our_eagle_cfg1_scale4_p20_flux/
base_dog_vid/                     our_eagle_cfg1_scale4_p5_flux/
base_dog_zimg/                    our_eifel_vid/
base_eifel_vid/                   our_hand_interpolate_vid/
base_liberty_st

In [68]:
pip install dreamsim


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 30.2 MB/s eta 0:00:00
  DEPRECATION: Building 'dreamsim' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'dreamsim'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for dreamsim: filename=dreamsim-0.2.1-py3-none-any.whl size=22277 sha256=60f4035922e45c3d0742ce6c657d893cacdd4ac1c3265ce7025dc139b9116f99
  Stored in directory: /home/nessessence/.cache/pip/wheels/2e/75/d8/e4b4c8bd53c360cab251bfa9639b54a76303a3e8e36dde94e3
Successfully built dreamsim
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dreamsim]
Note: you may need to restart the kernel to use updated pa

In [77]:
pip install lpips


Note: you may need to restart the kernel to use updated packages.
